# Parameter Permutation Sweeps with RasPermutation

Generate Cartesian products of parameter ranges, automatically
partition into batch folders (respecting the 99-plan HEC-RAS limit),
execute in parallel, and collect results into audit-friendly CSV logs.


In [ ]:
from ras_commander import *
from ras_commander import RasPermutation, RangeSpec

from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

plt.style.use("default")
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


## Setup & Project Extraction

Start with the reproducible `Muncie` example project, remove any old
permutation demo folders from prior runs, and inspect the existing
plans in `ras.plan_df`.


In [ ]:
project_path = RasExamples.extract_project("Muncie")
init_ras_project(project_path, "6.6")

suffix = "parameter_permutation_demo"
master_log_path = project_path.parent / (
    f"{project_path.name}_{suffix}_master_log.csv"
)

for batch_folder in RasPermutation.discover_batch_folders(
    project_path,
    suffix=suffix,
):
    shutil.rmtree(batch_folder, ignore_errors=True)

master_log_path.unlink(missing_ok=True)

available_slots = 99 - len(ras.plan_df)
plan_overview = (
    ras.plan_df[["plan_number", "Plan Title", "Short Identifier"]]
    .sort_values("plan_number")
    .reset_index(drop=True)
)

print(f"Project folder: {project_path}")
print(f"Existing plans: {len(ras.plan_df)}")
print(
    "Available new-plan slots before the 99-plan ceiling: "
    f"{available_slots}"
)
display(plan_overview)


## Define Parameters

`RasPermutation.define_parameters()` accepts plain value lists and
`RangeSpec` objects. Here the sweep uses three Manning's `n` values
and five width values, which yields a 15-row Cartesian product with
a stable `absolute_perm_id`.


In [ ]:
parameter_spec = {
    "mannings_n": [0.03, 0.04, 0.05],
    "width": RangeSpec(100, 300, 50),
}

width_values = parameter_spec["width"].generate_values()
parameters_df = RasPermutation.define_parameters(parameter_spec)

print(f"RangeSpec values for width: {width_values}")
print(
    f"{parameters_df['mannings_n'].nunique()} Manning's n values x "
    f"{parameters_df['width'].nunique()} widths = "
    f"{len(parameters_df)} permutations"
)
display(parameters_df)


## Cartesian Product Visualization

A compact way to see the sweep is to pivot the permutation table by
parameter values. Each heatmap cell corresponds to one row in the
permutation DataFrame, labeled by `absolute_perm_id`.


In [ ]:
grid = parameters_df.pivot(
    index="mannings_n",
    columns="width",
    values="absolute_perm_id",
)
display(grid)

fig, ax = plt.subplots(figsize=(10, 3.8))
image = ax.imshow(grid.values, cmap="Blues", aspect="auto")

ax.set_xticks(range(len(grid.columns)))
ax.set_xticklabels([int(value) for value in grid.columns])
ax.set_yticks(range(len(grid.index)))
ax.set_yticklabels([f"{value:.2f}" for value in grid.index])
ax.set_xlabel("Width parameter")
ax.set_ylabel("Manning's n")
ax.set_title(
    "Cartesian product grid (cell value = absolute_perm_id)"
)

for row_index in range(grid.shape[0]):
    for col_index in range(grid.shape[1]):
        ax.text(
            col_index,
            row_index,
            int(grid.iloc[row_index, col_index]),
            ha="center",
            va="center",
            color="black",
            fontsize=10,
        )

plt.colorbar(image, ax=ax, label="absolute_perm_id")
plt.show()


## Generate Plans

`RasPermutation.generate_plans()` clones the template plan, applies a
custom `apply_fn`, and writes both a master CSV and a per-batch CSV.
This notebook deliberately stops at plan generation and CSV tracking:
`execute_and_summarize()` is intentionally omitted because actual
HEC-RAS execution is slow.

HEC-RAS allows at most 99 plans per project. `Muncie` starts with 3
plans, so up to 96 new plans could fit in one cloned project. To make
the batch behavior visible in a small example, the demo sets
`max_plans_per_batch=10`, which forces 15 permutations into two batch
folders.


In [ ]:
def apply_description(plan_path, param_row, batch_ras):
    plan_number = plan_path.suffix[-2:]
    description = "\n".join(
        [
            "Parameter permutation demo",
            f"absolute_perm_id={int(param_row['absolute_perm_id'])}",
            f"mannings_n={param_row['mannings_n']:.2f}",
            f"width={param_row['width']:.1f}",
        ]
    )

    RasPlan.update_plan_description(
        plan_number,
        description,
        ras_object=batch_ras,
    )


plan_matrix = RasPermutation.generate_plans(
    template_plan="01",
    parameters_df=parameters_df,
    apply_fn=apply_description,
    suffix=suffix,
    max_plans_per_batch=10,
)

master_log_df = pd.read_csv(plan_matrix["master_log"])
first_batch_log_path = (
    plan_matrix["batch_folders"][0] / "permutations_log.csv"
)
first_batch_log_df = pd.read_csv(first_batch_log_path)

print(f"Master log: {plan_matrix['master_log'].name}")
print(
    "Batch folders: "
    f"{[folder.name for folder in plan_matrix['batch_folders']]}"
)
print(f"Total permutations: {plan_matrix['total_permutations']}")
print(f"Batch count: {plan_matrix['batches']}")
print()
print("Master log preview")
display(master_log_df.head())
print("First batch log preview")
display(first_batch_log_df.head())


## Batch Folder Organization

The master log tracks every permutation across all batch folders,
while each batch folder keeps its own local `permutations_log.csv`.
The tree preview below makes the folder layout explicit.


In [ ]:
effective_batch_size = min(99, 10, available_slots)
print(
    "Effective batch size = min(99 plan ceiling, "
    f"10 requested, {available_slots} available slots) = "
    f"{effective_batch_size}"
)
print()

master_preview = "\n".join(
    plan_matrix["master_log"].read_text(
        encoding="utf-8"
    ).splitlines()[:6]
)
batch_preview = "\n".join(
    first_batch_log_path.read_text(
        encoding="utf-8"
    ).splitlines()[:6]
)

print("Master CSV preview")
print(master_preview)
print()
print("Per-batch CSV preview")
print(batch_preview)


def render_batch_tree(batch_folder):
    lines = [f"{batch_folder.name}/"]
    for path in sorted(batch_folder.iterdir()):
        suffix = path.suffix.lower()
        is_plan_file = (
            len(suffix) == 4
            and suffix.startswith(".p")
            and suffix[2:].isdigit()
        )
        if path.is_dir() and path.name == "GIS_Data":
            lines.append(f"  {path.name}/")
        elif (
            path.name == "permutations_log.csv"
            or suffix == ".prj"
            or is_plan_file
        ):
            lines.append(f"  {path.name}")
    return "\n".join(lines)


for batch_folder in plan_matrix["batch_folders"]:
    print()
    print(render_batch_tree(batch_folder))

batch_counts = master_log_df.groupby("batch_folder").size()
fig, ax = plt.subplots(figsize=(8, 3.5))
bars = ax.bar(
    batch_counts.index,
    batch_counts.values,
    color=["#284b63", "#3c6e71"],
)
ax.axhline(
    99,
    color="#d9d9d9",
    linestyle="--",
    linewidth=1,
    label="HEC-RAS 99-plan ceiling",
)
ax.set_ylabel("Permutations in batch")
ax.set_title("Generated permutations by batch folder")
ax.legend()

for bar, count in zip(bars, batch_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.2,
        str(int(count)),
        ha="center",
        va="bottom",
    )

plt.xticks(rotation=0)
plt.show()


## Discover Batch Folders

`RasPermutation.discover_batch_folders()` is useful when you want to
resume work from an existing sweep folder without regenerating plans.
It finds folders that follow the `{project}_{suffix}_{NNN}` naming
convention and returns them in numeric order.


In [ ]:
discovered_batches = RasPermutation.discover_batch_folders(
    project_path,
    suffix=suffix,
)

print("Discovered batch folders:")
for folder in discovered_batches:
    print(f"- {folder}")

batch_summary = (
    master_log_df.groupby("batch_folder")["absolute_perm_id"]
    .agg(["min", "max", "count"])
)
display(batch_summary)

assert discovered_batches == plan_matrix["batch_folders"]


## Key Takeaways

- Use `RasPermutation.define_parameters()` to turn lists and
  `RangeSpec` objects into a reproducible Cartesian product table.
- `apply_fn` is the extensibility hook. This example only updates the
  plan description, but the same pattern can edit geometry, flow, or
  other plan-linked files before execution.
- `absolute_perm_id` keeps every permutation traceable across the
  master log and the per-batch logs.
- `generate_plans()` handles project cloning and batching so the
  workflow stays within the HEC-RAS 99-plan limit.
- `discover_batch_folders()` makes it easy to pick the sweep back up
  later without rebuilding the folder tree.
